# step00a - QC/QA: Filter before/after (isolated)

Quantifies what the low-pass **Filter** actually does to each cast, before Loop Edit. Companion to
step00b (Filter + Loop Edit panels); this one isolates the Filter and reports the numbers you would
quote in a methods section.

For each cast it:
1. runs `FilterW.exe` on the aligned cnv (the same PSA the pipeline uses);
2. computes, for pressure (and temperature / conductivity if present), the scan-to-scan change
   dX/dt = gradient(X) and its standard deviation **before** vs **after** filtering;
3. reports `reduction% = 100 * (1 - std_filtered / std_before)`.

For a strain-gauge SBE 19plus V2 the pressure reduction is expected to be small (little dithering to
remove), unlike a 9plus / Digiquartz. Variables the PSA does not filter show ~0%.

Runs on **every aligned cast in the folder** (`ALIGN_DIR/*_al.cnv`); set `CASTS_OVERRIDE` to limit.
Pressure runs surface (0) at the top to bottom below on every figure.

> **Zoom / interactive:** the batch figures (section 4) render as reliable static PNGs (also saved to
> `qc_plots/`). Section 5 adds an interactive explorer that draws one cast/variable at a time as a
> live, mouse drag/scroll zoomable canvas (`INTERACTIVE_ZOOM = True`, the default; needs `ipympl`:
> `pip install ipympl`). Set `INTERACTIVE_ZOOM = False` for a fully static run.

QA tool only: it does not change the L1/L2 products. (The headless `step00a_filter_qa.py` remains for
batch/command-line runs.)


## 1. Imports, style, settings


In [ ]:
from __future__ import annotations
import re, subprocess
import xml.etree.ElementTree as ET
from pathlib import Path
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Display backend.
#   INTERACTIVE_ZOOM = True (default): the section 5 explorer renders a live ipympl canvas you can
#       mouse drag/scroll zoom (needs ipympl: pip install ipympl). The batch figures (section 4)
#       always render as reliable static PNGs regardless, so a many-figure run never goes blank.
#   INTERACTIVE_ZOOM = False: everything static inline; the explorer still redraws on dropdown
#       change, just without mouse zoom.
# Decided BEFORE pyplot is imported / a backend is chosen.
# ---------------------------------------------------------------------------
INTERACTIVE_ZOOM = True

import matplotlib
_backend = "widget" if INTERACTIVE_ZOOM else "inline"
try:
    get_ipython().run_line_magic("matplotlib", _backend)
    if INTERACTIVE_ZOOM:
        print("matplotlib: ipympl widget backend active - the section 5 explorer has mouse drag/scroll zoom.")
except Exception:
    if INTERACTIVE_ZOOM:
        print("ipympl not available -> explorer will be static (no mouse zoom). To enable it:")
        print("    pip install ipympl        (or)        conda install -c conda-forge ipympl")
        print("    then: Restart Kernel + Run All")
    INTERACTIVE_ZOOM = False
    try:
        get_ipython().run_line_magic("matplotlib", "inline")
    except Exception:
        try: matplotlib.use("Agg")
        except Exception: pass

import matplotlib.pyplot as plt
from matplotlib import rcParams
try:
    from IPython.display import display, Image
except Exception:
    def display(obj): print(obj)
    Image = None

rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "axes.titlesize": 10,
                 "axes.labelsize": 9, "axes.linewidth": 0.8, "xtick.direction": "in",
                 "ytick.direction": "in", "xtick.labelsize": 8, "ytick.labelsize": 8,
                 "legend.fontsize": 7.5, "axes.grid": False, "font.family": "DejaVu Sans"})
C_BEFORE, C_FILTER = "#E8703A", "#2E6FB0"


def _despine(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)


def finish(fig, out):
    """Save the figure, then show it as a reliable STATIC PNG snapshot (never blank, which matters
    for many batch figures). Live mouse-zoom is provided by the section 5 explorer via show_live()."""
    saved = False
    for kw in ({"bbox_inches": "tight"}, {}):
        try:
            fig.savefig(out, **kw); saved = True; break
        except Exception as e:
            print(f"    savefig {'tight' if kw else 'plain'} failed for {out.name}: {e}")
    if saved and Image is not None:
        try: display(Image(filename=str(out)))
        except Exception: display(fig)
    else:
        display(fig)
    plt.close(fig)
    return out if saved else None


_EXPLORER_FIGS = {}
def show_live(fig, key):
    """Live ipympl canvas when INTERACTIVE_ZOOM is on (mouse box-zoom, scroll zoom, Home to reset),
    else a static inline PNG. Closes the previous figure for this key to avoid memory pile-up."""
    if INTERACTIVE_ZOOM:
        prev = _EXPLORER_FIGS.get(key)
        if prev is not None:
            try: plt.close(prev)
            except Exception: pass
        _EXPLORER_FIGS[key] = fig
        for attr, val in (("header_visible", False), ("footer_visible", False)):
            try: setattr(fig.canvas, attr, val)
            except Exception: pass
        display(fig.canvas)
    else:
        plt.show(); plt.close(fig)


In [ ]:
# ===========================================================================
# Paths / settings
# ===========================================================================
CTD_ROOT   = Path(r"C:\Projects\ctd_pipeline")
CRUISE     = "P45_06"
SBE_BIN    = Path(r"C:\Program Files (x86)\Sea-Bird\SBEDataProcessing-Win32")
PSA_DIR    = CTD_ROOT / "psa"
ALIGN_DIR  = CTD_ROOT / "cruises" / CRUISE / "L2" / "CTD" / "Align_CTD"
FILTER_PSA = PSA_DIR / "05_filter.psa"
WORK       = CTD_ROOT / "_sbe_work" / "qc"; WORK.mkdir(parents=True, exist_ok=True)
QC_PLOT_DIR = CTD_ROOT / "cruises" / CRUISE / "L2" / "CTD" / "qc_plots"; QC_PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Casts to process. Default: EVERY aligned cast in ALIGN_DIR. To limit to a quick subset, list ids
# in CASTS_OVERRIDE, e.g. ["P45_06_CTD_03", "P45_06_CTD_06"].
CASTS_OVERRIDE = []
CASTS = CASTS_OVERRIDE or sorted(p.name[:-len("_al.cnv")] for p in ALIGN_DIR.glob("*_al.cnv"))

# variables to test: (cnv short-name candidates, label)
VARS = [(("prdM", "prDM", "prM"), "pressure"),
        (("tv290C", "t090C", "tv290"), "temperature"),
        (("c0S/m", "cond0S/m", "c0mS/cm", "c_S/m"), "conductivity")]
_VAR_DXLABEL = {"pressure": "dP/dt (db/scan)", "temperature": "dT/dt (\u00b0C/scan)",
                "conductivity": "dC/dt (S/m/scan)"}
_VAR_XLABEL  = {"pressure": "pressure (db)", "temperature": "temperature (\u00b0C)",
                "conductivity": "conductivity (S/m)"}

print("Cruise:", CRUISE, "| casts:", len(CASTS), "found | filter PSA:", FILTER_PSA.name,
      "| zoom:", INTERACTIVE_ZOOM)
if not CASTS:
    print("  no *_al.cnv in", ALIGN_DIR, "- run step01 (align) first, or set CASTS_OVERRIDE.")


## 2. Readers and filter helpers


In [ ]:
def read_cnv(path):
    """Parse a .cnv keeping all scans."""
    if path is None: return None
    txt = Path(path).read_text(encoding="latin-1", errors="replace").splitlines()
    names, ds = {}, None
    nr = re.compile(r"#\s*name\s+(\d+)\s*=\s*([^:]+?)\s*[:=]", re.I)
    for i, l in enumerate(txt):
        s = l.strip()
        if s.startswith("#") or s.startswith("*"):
            m = nr.search(l)
            if m: names[int(m.group(1))] = m.group(2).strip()
            if s.upper().startswith("*END*"): ds = i + 1; break
    if ds is None: return None
    cols = [names[k] for k in sorted(names)]
    rows = [l.split()[:len(cols)] for l in txt[ds:] if l.strip() and len(l.split()) >= len(cols)]
    return pd.DataFrame(rows, columns=cols).apply(pd.to_numeric, errors="coerce")


def pick(df, cands):
    if df is None: return None
    for c in cands:
        if c in df.columns: return c
    for c in df.columns:
        if any(x.lower() in c.lower() for x in cands): return c
    return None


def describe_filter_psa(psa):
    """Read the PSA and return (list of (variable, filter-label), summary string)."""
    root = ET.parse(psa).getroot()
    def tc(tag):
        el = root.find(tag)
        return float(el.get("value")) if el is not None else None
    tcA, tcB = tc("TimeConstFilterA"), tc("TimeConstFilterB")
    names = [ci.find("Calc/FullName").get("value") for ci in root.find("CalcArray")]
    types = [int(ai.get("value")) for ai in root.find("FilterTypeArray")]
    label = {0: "none", 1: f"low-pass A ({tcA:g} s)", 2: f"low-pass B ({tcB:g} s)"}
    rows = [(n, label.get(t, f"type {t}")) for n, t in zip(names, types)]
    applied = [f"{n} -> {lab}" for n, lab in rows if not lab.startswith("none")]
    summ = "; ".join(applied) if applied else "nothing (all variables set to no filter)"
    return rows, summ


def run_filter(in_cnv, stem):
    subprocess.run([str(SBE_BIN / "FilterW.exe"), f"/p{FILTER_PSA}", f"/i{in_cnv}",
                    f"/o{WORK}", f"/f{stem}", "/s"], capture_output=True, text=True)
    out = WORK / f"{stem}.cnv"
    return out if out.exists() else None


def dxdt_std(series):
    """Std of the scan-to-scan change of a series (the high-frequency content the filter removes)."""
    x = pd.to_numeric(series, errors="coerce").to_numpy(float)
    x = x[np.isfinite(x)]
    if x.size < 3: return np.nan
    return float(np.nanstd(np.gradient(x)))


## 3. Plot: before/after profile + dX/dt (shared by batch and explorer)


In [ ]:
def _render_filter(fig, axes, cast, before, filt, cands, label):
    """Draw before/after for one variable onto (axes[0], axes[1]):
       panel 0 = variable vs pressure ; panel 1 = dX/dt vs pressure (the content the filter removes).
       Pressure runs surface (top) to bottom. Returns (std_before, std_filtered, reduction%)."""
    pcb = pick(before, VARS[0][0]); pcf = pick(filt, VARS[0][0])
    b = pick(before, cands); f = pick(filt, cands)
    if pcb is None or b is None:
        return (np.nan, np.nan, np.nan)
    Pb = pd.to_numeric(before[pcb], errors="coerce")
    Vb = pd.to_numeric(before[b], errors="coerce")
    axes[0].plot(Vb, Pb, lw=0.5, color=C_BEFORE, label="before", alpha=0.9)
    axes[1].plot(np.gradient(Vb.to_numpy(float)), Pb, lw=0.5, color=C_BEFORE, label="before", alpha=0.9)
    sb = dxdt_std(before[b]); sf = np.nan
    if filt is not None and f is not None and pcf is not None:
        Pf = pd.to_numeric(filt[pcf], errors="coerce"); Vf = pd.to_numeric(filt[f], errors="coerce")
        axes[0].plot(Vf, Pf, lw=0.5, color=C_FILTER, label="filtered", alpha=0.9)
        axes[1].plot(np.gradient(Vf.to_numpy(float)), Pf, lw=0.5, color=C_FILTER, label="filtered", alpha=0.9)
        sf = dxdt_std(filt[f])
    red = 100 * (1 - sf / sb) if (sb and np.isfinite(sb) and sb > 0 and np.isfinite(sf)) else np.nan
    for a in axes: _despine(a)
    axes[0].invert_yaxis()                    # sharey -> invert once: surface at top
    axes[0].set_ylabel("pressure (db)")
    axes[0].set_xlabel(_VAR_XLABEL.get(label, label)); axes[0].set_title(f"{label}")
    axes[1].set_xlabel(_VAR_DXLABEL.get(label, "dX/dt")); axes[1].axvline(0, color="k", lw=0.5, alpha=0.3)
    rtxt = f"{red:.1f}%" if np.isfinite(red) else "n/a"
    axes[1].set_title(f"{label} dX/dt  (reduction {rtxt})")
    axes[0].legend(loc="lower left", fontsize=7.5)
    return (sb, sf, red)


def plot_filter_pressure(cast, before, filt):
    """Static per-cast figure: pressure profile + dP/dt, before vs filtered."""
    fig, axes = plt.subplots(1, 2, figsize=(8.4, 6.0), sharey=True)
    _render_filter(fig, axes, cast, before, filt, VARS[0][0], "pressure")
    fig.suptitle(f"{cast}: Filter before vs after (pressure)", fontsize=11)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    return finish(fig, QC_PLOT_DIR / f"{cast}__qc_filter_dpdt.png")


## 4. Run all casts
Runs Filter on every cast, prints the reduction table, writes `qc_filter_dpdt_summary.csv`, and saves
a per-cast pressure figure. Populates `ALL` for the explorer.


In [ ]:
rows_psa, filt_summary = describe_filter_psa(FILTER_PSA)
print(f"Filter PSA: {FILTER_PSA.name}")
print(f"  applied: {filt_summary}\n")

summary_rows = []; ALL = {}
hdr = f"{'cast':<16} {'variable':<13} {'std before':>12} {'std filtered':>13} {'reduction':>11}"
print(hdr); print("-" * len(hdr))
for cast in CASTS:
    aligned = ALIGN_DIR / f"{cast}_al.cnv"
    if not aligned.exists():
        print(f"{cast}: no aligned cnv"); continue
    before = read_cnv(aligned)
    filt = read_cnv(run_filter(aligned, f"{cast}_filt"))
    if before is None or filt is None:
        print(f"{cast}: filter run failed"); continue
    ALL[cast] = {"before": before, "filtered": filt}
    for cands, label in VARS:
        b = pick(before, cands); f = pick(filt, cands)
        if b is None or f is None:
            continue
        sb, sf = dxdt_std(before[b]), dxdt_std(filt[f])
        red = 100 * (1 - sf / sb) if (sb and np.isfinite(sb) and sb > 0) else np.nan
        print(f"{cast:<16} {label:<13} {sb:>12.4f} {sf:>13.4f} {red:>10.1f}%")
        summary_rows.append({"cast": cast, "variable": label,
                             "dxdt_std_before": round(sb, 5), "dxdt_std_filtered": round(sf, 5),
                             "reduction_pct": round(red, 1) if np.isfinite(red) else ""})
    png = plot_filter_pressure(cast, before, filt)
    if png: print(f"{'':<16} plot -> {png.name}")

if summary_rows:
    sdf = pd.DataFrame(summary_rows)
    out_csv = QC_PLOT_DIR / "qc_filter_dpdt_summary.csv"
    sdf.to_csv(out_csv, index=False, encoding="utf-8-sig")
    display(sdf); print(f"summary -> {out_csv}")

print("\nReduction% = 100 * (1 - std(dX/dt)_filtered / std(dX/dt)_before). A small pressure reduction")
print("is expected on a 19plus V2 (strain-gauge): little dithering to remove. Variables the PSA does")
print("not filter show ~0% (before and filtered identical).")


## 5. Interactive explorer (optional): one cast/variable at a time, zoomable
Pick a **cast** and **variable** (pressure / temperature / conductivity); the before-vs-filtered
profile and dX/dt redraw live. With `INTERACTIVE_ZOOM = True` and ipympl it is a live canvas (drag to
box-zoom, scroll to zoom, toolbar Home to reset). The "P zoom (db)" slider is a depth preset. The
diagnostics line prints std(dX/dt) before/filtered and the reduction %. Diagnostic only: writes no
files. Without ipympl it still redraws on dropdown change, just without mouse zoom.


In [ ]:
# ---- interactive explorer: Filter before/after, one cast/variable at a time, zoomable ----
try:
    import ipywidgets as W
    from IPython.display import clear_output
    _HAVE_W = True
except Exception:
    _HAVE_W = False
    print("ipywidgets not installed. The batch figures above are the static output; for the live",
          "explorer do: pip install ipywidgets ipympl, then Restart Kernel + Run All.")

def _cast_pmax(rec):
    b = rec.get("before")
    pc = pick(b, VARS[0][0]) if b is not None else None
    if pc is None: return 100.0
    return float(np.nanmax(pd.to_numeric(b[pc], errors="coerce")))

if _HAVE_W and "ALL" in dir() and ALL:
    _casts = sorted(ALL)
    _pmax = float(np.ceil(max(_cast_pmax(ALL[c]) for c in _casts) / 5.0) * 5.0)
    _varopts = [label for _c, label in VARS]
    cast_dd = W.Dropdown(options=_casts, value=_casts[0], description="cast")
    var_dd  = W.Dropdown(options=_varopts, value="pressure", description="variable")
    pr_sl   = W.FloatRangeSlider(value=[0.0, _pmax], min=0.0, max=_pmax, step=2.0,
                                 description="P zoom (db)", continuous_update=False,
                                 readout=True, readout_format=".0f")
    rst_bt  = W.Button(description="reset zoom")
    out = W.Output()
    _CANDS = {label: cands for cands, label in VARS}

    def _reset(_): pr_sl.value = [0.0, _pmax]
    rst_bt.on_click(_reset)

    def _draw(*_):
        with out:
            clear_output(wait=True)
            cast = cast_dd.value; rec = ALL[cast]; label = var_dd.value
            before, filt = rec.get("before"), rec.get("filtered")
            fig, axes = plt.subplots(1, 2, figsize=(9.0, 6.0), sharey=True)
            sb, sf, red = _render_filter(fig, axes, cast, before, filt, _CANDS[label], label)
            rtxt = f"{red:.1f}%" if np.isfinite(red) else "n/a"
            print(f"{cast} [{label}]: std(dX/dt) before={sb:.5f}, filtered={sf:.5f}, reduction={rtxt}")
            pm = _cast_pmax(rec)
            ylo = max(0.0, pr_sl.value[0]); yhi = min(pr_sl.value[1], pm)
            if yhi > ylo: axes[0].set_ylim(yhi, ylo)
            fig.suptitle(f"{cast}: Filter before vs after ({label})", fontsize=11)
            fig.tight_layout(rect=[0, 0, 1, 0.97])
            show_live(fig, "filter")

    for _w in (cast_dd, var_dd, pr_sl): _w.observe(_draw, "value")
    display(W.VBox([W.HBox([cast_dd, var_dd]), W.HBox([pr_sl, rst_bt]), out])); _draw()
elif _HAVE_W:
    print("Run section 4 (Run all casts) first so ALL is populated.")
